In [ ]:
import os
import re
import traceback
import subprocess
import re
def extract_python_blocks(text: str) -> list[str]:
    """
    从传入的 Markdown 字符串中提取所有以 ```python 开头、``` 结尾的代码块。
    返回一个去除首尾空白的代码片段列表。
    """
    pattern = re.compile(r"```python\s*(.*?)\s*```", re.DOTALL)
    code_blocks = '\n\n'.join(pattern.findall(text))

    return code_blocks

def process_code_snippets(item: dict, output_dir: str = "/fs-computility/mllm1/shared/hub/datasets--xxxllz--Chart2Code-160k/processed_code") -> tuple[dict, bool]:
    """
    处理代码片段，执行以下操作：
    1. 将原始代码存储成一个 .py 文件，并检查其可运行性。
    2. 如果原始代码可运行，则修改代码：
        a. 将 plt.savefig 后面的存储路径提取出来，改成 id_val + '_generated.pdf'。
        b. 删除 plt.close()。
    3. 重新执行修改后的代码，如果可运行则保留修改后的代码。

    Args:
        item (dict): 包含 'conversations' 键的字典，其中包含原始的 GPT 输出。
        id_val (str): 用于文件命名和图片生成路径的唯一ID。
        output_dir (str): 保存处理后的代码文件的目录。

    Returns:
        tuple[dict, bool]: 一个元组，第一个元素是包含处理结果的字典（如果成功则包含 modified_code），
                            第二个元素是一个布尔值，表示操作是否全部成功 (True) 或失败 (False)。
    """
    
    org_gpt_output = item['conversations'][-1]['value']
    id_val = item['id']
    original_code = extract_python_blocks(org_gpt_output)

    if not original_code:
        print(f"[ID: {id_val}] 未能从 GPT 输出中提取到 Python 代码块。")
        return {}, False

    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    original_filepath = os.path.join(output_dir, f"{id_val}_original.py")
    modified_filepath = os.path.join(output_dir, f"{id_val}_modified.py")
    
    results = {"id": id_val, "image": item['image'], "conversations": [item['conversations'][0]]}

    # 步骤 1: 将原始代码存储成一个py文件，并检查其可运行性
    try:
        with open(original_filepath, "w", encoding="utf-8") as f:
            f.write(original_code)
        
        print(f"[ID: {id_val}] 尝试运行原始代码...")
        run_result = subprocess.run(
            ["python", original_filepath],
            capture_output=True,
            text=True,
            timeout=10
        )

        if run_result.returncode != 0:
            print(f"[ID: {id_val}] 原始代码运行失败。错误：\n{run_result.stderr}")
            if os.path.exists(original_filepath):
                os.remove(original_filepath)
            return results, False
        
        print(f"[ID: {id_val}] 原始代码运行成功。")

    except Exception as e:
        print(f"[ID: {id_val}] 保存或运行原始代码时发生异常：{e}")
        if os.path.exists(original_filepath):
            os.remove(original_filepath)
        return results, False

    # 步骤 2 & 3: 修改代码 (savefig路径, 删除plt.close)
    modified_code = original_code

    # 步骤 2a: 留下有 plt.savefig 的，把 plt.savefig 后面的存储路径提取出来，改成 id_val + '_generated.pdf'
    if "plt.savefig" in modified_code:
        print(f"[ID: {id_val}] 找到 plt.savefig，正在修改路径...")
        modified_code = re.sub(r"plt\.savefig\(['\"].*?['\"]\)", 
                               f"plt.savefig('{id_val}_generated.pdf')", 
                               modified_code)
        print(f"[ID: {id_val}] plt.savefig 路径已修改为 '{id_val}_generated.pdf'")
    else:
        print(f"[ID: {id_val}] 未找到 plt.savefig。此操作失败，因为要求保留有 plt.savefig 的。")
        # 如果没有 plt.savefig，并且我们要求只保留有它的，那么这里算作失败
        return results, False # 未找到 plt.savefig 视为失败

    # 步骤 2b: 将 plt.close() 进行删除 (无论是否存在都不算失败)
    if "plt.close()" in modified_code:
        print(f"[ID: {id_val}] 找到 plt.close()，正在删除...")
        modified_code = modified_code.replace("plt.close()", "")
        print(f"[ID: {id_val}] plt.close() 已删除。")
    else:
        print(f"[ID: {id_val}] 未找到 plt.close()。")

    # 保存修改后的代码
    try:
        with open(modified_filepath, "w", encoding="utf-8") as f:
            f.write(modified_code)
    except Exception as e:
        print(f"[ID: {id_val}] 保存修改后的代码时发生异常：{e}")
        return results, False
    
    # 步骤 3: 重新执行修改后的这个代码，保留能运行的
    try:
        print(f"[ID: {id_val}] 尝试运行修改后的代码...")
        run_result_modified = subprocess.run(
            ["python", modified_filepath],
            capture_output=True,
            text=True,
            timeout=10
        )

        if run_result_modified.returncode != 0:
            print(f"[ID: {id_val}] 修改后的代码运行失败。错误：\n{run_result_modified.stderr}")
            if os.path.exists(modified_filepath):
                os.remove(modified_filepath)
            return results, False
        
        print(f"[ID: {id_val}] 修改后的代码运行成功。")

    except Exception as e:
        print(f"[ID: {id_val}] 运行修改后的代码时发生异常：{e}")
        if os.path.exists(modified_filepath):
            os.remove(modified_filepath)
        return results, False
            
    # 如果所有步骤都成功，返回修改后的代码和 True
    results["conversations"].append({
        "from": "gpt",
        "value": "```python " + modified_code + " ```"
    })
    return results, True

In [10]:
import subprocess
import time
import os
import signal
import psutil

def execute_python_with_timeout(code_file, timeout=100, memory_limit_mb=500):
    code_log_texts_file = code_file.replace(".py", "_log_chart_types.py")
    print(f"Executing: {code_log_texts_file}")

    # 构建命令行调用 python3 文件
    command = ["python3", code_log_texts_file]

    # 创建子进程对象，启动新会话
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE, start_new_session=True)
    proc = psutil.Process(process.pid)

    start_time = time.time()

    try:
        while True:
            time.sleep(1)
            elapsed_time = time.time() - start_time

            # 检查超时
            if elapsed_time > timeout:
                print(f"Process exceeded timeout of {timeout} seconds. Terminating.")
                os.killpg(os.getpgid(process.pid), signal.SIGTERM)
                break

            # 检查内存使用
            memory_usage = proc.memory_info().rss / (1024 * 1024)  # 转换为 MB
            if memory_usage > memory_limit_mb:
                print(f"Process exceeded memory limit of {memory_limit_mb} MB (Current: {memory_usage:.2f} MB). Terminating.")
                os.killpg(os.getpgid(process.pid), signal.SIGTERM)
                break

            # 检查进程是否已结束
            if process.poll() is not None:
                break

        # 等待进程结束并获取输出
        stdout, stderr = process.communicate()
        print("Process output:")
        print(stdout.decode())
        if stderr:
            print("Process errors:")
            print(stderr.decode())

    except Exception as e:
        print(f"An error occurred: {e}")
        os.killpg(os.getpgid(process.pid), signal.SIGTERM)

    finally:
        # 确保进程被终止
        if process.poll() is None:
            os.killpg(os.getpgid(process.pid), signal.SIGKILL)
            print("Process killed.")

# 示例调用函数
code_file = "/fs-computility/mllm1/fangxinyu/plot2code/verl/outputs/verl_generated_plot_in_reward/890_generated_by_model_in_training.py"
execute_python_with_timeout(code_file, timeout=100, memory_limit_mb=500)


Executing: /fs-computility/mllm1/fangxinyu/plot2code/verl/outputs/verl_generated_plot_in_reward/890_generated_by_model_in_training_log_chart_types.py
Process exceeded memory limit of 500 MB (Current: 512.73 MB). Terminating.
Process output:



In [4]:
import json

# LOAD & DUMP
def dump(data, f, **kwargs):

    def dump_json(data, pth, **kwargs):
        json.dump(data, open(pth, 'w'), indent=4, ensure_ascii=False)

    def dump_xlsx(data, f, **kwargs):
        data.to_excel(f, index=False, engine='xlsxwriter')


    handlers = dict(json=dump_json, xlsx=dump_xlsx)
    suffix = f.split('.')[-1]
    return handlers[suffix](data, f, **kwargs)


def load(f, fmt=None):

    def load_json(pth):
        return json.load(open(pth, 'r', encoding='utf-8'))

    def load_jsonl(f):
        lines = open(f, encoding='utf-8').readlines()
        lines = [x.strip() for x in lines]
        if lines[-1] == '':
            lines = lines[:-1]
        data = [json.loads(x) for x in lines]
        return data

    handlers = dict(json=load_json, jsonl=load_jsonl)
    if fmt is not None:
        return handlers[fmt](f)

    suffix = f.split('.')[-1]
    return handlers[suffix](f)

meta = load('/fs-computility/mllm1/shared/hub/datasets--xxxllz--Chart2Code-160k/chart2code_160k.json')
filtered_data = []
for item in meta:
    code, flag = process_code_snippets(item)
    if flag:
        print(f"[ID: {item['id']}] 处理成功，添加到结果列表。")
        filtered_data.append(code)

[ID: 127] 修改后的代码运行成功。
[ID: 127] 处理成功，添加到结果列表。
[ID: 128] 尝试运行原始代码...
[ID: 128] 原始代码运行成功。
[ID: 128] 找到 plt.savefig，正在修改路径...
[ID: 128] plt.savefig 路径已修改为 '128_generated.pdf'
[ID: 128] 未找到 plt.close()。
[ID: 128] 尝试运行修改后的代码...
[ID: 128] 修改后的代码运行成功。
[ID: 128] 处理成功，添加到结果列表。
[ID: 129] 尝试运行原始代码...
[ID: 129] 原始代码运行成功。
[ID: 129] 找到 plt.savefig，正在修改路径...
[ID: 129] plt.savefig 路径已修改为 '129_generated.pdf'
[ID: 129] 未找到 plt.close()。
[ID: 129] 尝试运行修改后的代码...
[ID: 129] 修改后的代码运行成功。
[ID: 129] 处理成功，添加到结果列表。
[ID: 130] 尝试运行原始代码...


KeyboardInterrupt: 

In [5]:
filtered_data

[{'id': 1,
  'image': 'images/1.png',
  'conversations': [{'from': 'human',
    'value': '<image>\nYou are an expert developer specializing in writing Python matplotlib code based on a given picture.    I need your help to generate the Python code that can reproduce the picture based on the picture I provided.\nTo ensure accuracy and detail in your recreation, you need to begin with a comprehensive analysis of the figure.    You should generate code snippets with the following steps\n1.Layout and Chart Type Analysis: e.g., identify the picture’s composition, noting the presence,arrangement of any subplots and how many charts are within a subplot.\n2.Data Analysis: e.g., summarize the data trend or pattern.\n3.Additional Features: e.g., identify any supplementary elements such as legends, colormaps, tick labels, or text annotations that contribute to the figure’s clarity or aesthetic appeal.\n4.Then generate the final code according to the previous analysis.'},
   {'from': 'gpt',
    'v

In [3]:
import os
import subprocess

code = extract_python_blocks(answer).replace('import', 'xxx')

code_log_texts_file = "xxx.py"
with open(code_log_texts_file, 'w') as f:
    f.write(code)

try:
    # 使用 subprocess 来执行 Python 脚本并捕获错误
    result = subprocess.run(f"python3 {code_log_texts_file}", shell=True, check=True, capture_output=True, text=True)
    print(result.stdout)  # 输出正常的标准输出
except subprocess.CalledProcessError as e:
    print(f"Error occurred while running the script: {e}")
    print(f"Error output: {e.stderr}")


Error occurred while running the script: Command 'python3 xxx.py' returned non-zero exit status 1.
Error output:   File "/fs-computility/mllm1/fangxinyu/plot2code/verl/xxx.py", line 1
    xxx matplotlib.pyplot as plt
        ^^^^^^^^^^
SyntaxError: invalid syntax



In [ ]:
import sys
import os
os.environ['PROJECT_PACK_PATH']='/fs-computility/mllm1/fangxinyu/plot2code/verl/verl/utils/reward_score/chart2code'
sys.path.insert(0,os.environ['PROJECT_PACK_PATH'])
from evaluator.text_evaluator import TextEvaluator
from evaluator.chart_type_evaluator import ChartTypeEvaluator
from evaluator.legend_evaluator import LegendEvaluator
from evaluator.grid_evaluator import GridEvaluator
from evaluator.color_evaluator import ColorEvaluator
from evaluator.layout_evaluator import LayoutEvaluator

: 

In [1]:
import sys
import os
os.environ['PROJECT_PACK_PATH']='/fs-computility/mllm1/fangxinyu/plot2code/verl/verl/utils/reward_score/chart2code'
sys.path.insert(0,os.environ['PROJECT_PACK_PATH'])
from evaluator.text_evaluator import TextEvaluator
from evaluator.layout_evaluator import LayoutEvaluator

In [4]:
generated_py_file = '/fs-computility/mllm1/fangxinyu/plot2code/verl/outputs/verl_generated_plot_in_reward/14572_gt_org.py'
original_py_file = '/fs-computility/mllm1/fangxinyu/plot2code/verl/outputs/verl_generated_plot_in_reward/14572_gt_org.py'
os.environ['PROJECT_STORE_PATH']='/fs-computility/mllm1/fangxinyu/plot2code/verl/outputs/verl_generated_plot_in_reward'
# text_evaluator = TextEvaluator(use_position=False, use_axs=False)
# text_value = text_evaluator(generation_code_file=generated_py_file, golden_code_file=original_py_file)
text_evaluator = LayoutEvaluator()
layout_value = text_evaluator(generation_code_file=generated_py_file, golden_code_file=original_py_file)
print(layout_value)

None
